# 4b · Do IQB scores track external indicators? (concurrent validity)

**Research question (RQ 4b):** *To what extent do IQB scores correlate with financial indicators, market concentration indicators such as the HHI, or other socio-economic indicators, thereby establishing the validity of this framework as an effective tool in policy decisions?* Analytical approach: *compute correlations between IQB scores and external indicators (GDP per capita, ITU broadband penetration, ISP HHI) to assess concurrent validity.*

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

import iqb_tools as T

T.set_style()
cfg = T.config_table()
ctx = T.read_table("country_context").set_index("country_code")
d = ctx[T.binding_tests(ctx) >= T.MIN_TESTS].copy()
d["income"] = pd.Categorical(d.income_group, T.INCOME_ORDER, ordered=True)

# 2025 pass rates per country: the test-weighted average of the 12 months
df25 = T.load_level("country", months=pd.date_range("2025-01-01", "2025-12-01", freq="MS"))
tests = T.distinct_tests(cfg)
C_all = T.pool_pass_rates(df25, T.pass_rates(df25, tests), "country_code", tests)  # every country (see the last section)
C = C_all.loc[d.index]
S = T.iqb_scores(C, cfg, T.Q_GRID)  # IQB score of each country at P1..P99

d["median_download"] = df25.groupby("country_code").download_p50.median()
d["median_latency"] = df25.groupby("country_code").latency_p50.median()
d["log10_tests"] = np.log10(d.n_down)

INDICATORS = {
    "log10_gdp_pc": "GDP per capita (log, PPP)",
    "fixed_bb_per100": "Fixed broadband per 100 (ITU)",
    "internet_users_pct": "Internet users, % (ITU)",
    "mobile_subs_per100": "Mobile subscriptions per 100 (ITU)",
    "secure_servers_per_m": "Secure servers per million",
    "isp_hhi": "ISP HHI (APNIC)",
}
d[list(INDICATORS)].notna().sum().rename(INDICATORS).to_frame("countries with value")

The sample here is every country with at least 1,153 tests in 2025, 211 in all, including 10 without a World Bank income group, so it is larger than the 201 used in the other notebooks. The table counts how many of them have a value for each external indicator (assembled in 0-setup). Each correlation below uses only the countries with a value, so their number varies from 176 (Internet users) to 210 (ISP HHI).

## How strongly does IQB track the indicators?

In [ ]:
measures = {f"IQB P{q}": S[q] for q in (25, 50, 75, 95)}
measures["IQB mean over P1–P99"] = S.mean(axis=1)
measures["median download (Mbit/s)"] = d.median_download
measures["median latency (negated)"] = -d.median_latency

rows = []
for mname, x in measures.items():
    for ind, label in INDICATORS.items():
        rho, lo, hi, n = T.spearman_ci(x, d[ind])
        rows.append((mname, label, rho, lo, hi, n))
corr = pd.DataFrame(rows, columns=["measure", "indicator", "rho", "lo", "hi", "n"])
tbl = corr.assign(
    cell=lambda x: (
        x.rho.map("{:.2f}".format)
        + " ["
        + x.lo.map("{:.2f}".format)
        + ", "
        + x.hi.map("{:.2f}".format)
        + "]"
    )
)
tbl.pivot(index="measure", columns="indicator", values="cell").loc[
    list(measures), list(INDICATORS.values())
]


Each cell is the Spearman rank correlation (ρ) between a measure of connection quality (rows) and an external indicator (columns), across the countries that have both, with its 95% confidence interval in brackets. The rows are IQB at four percentiles, IQB averaged over P1–P99, and two plain measures for comparison: the median download speed and the median latency (negated, so that higher means better in every row). A ρ of 1 would mean that a measure ranks countries exactly as the indicator does, and 0 means no relation; an interval that excludes 0 means the relation is unlikely to be chance.

The correlations are strongest with GDP per capita, fixed broadband, Internet use and secure servers (0.52–0.80), weaker with mobile subscriptions (0.31–0.48), and weak and negative with ISP concentration (−0.07 to −0.30). For the four strongest indicators, IQB correlates best at P50 and worst at P95 among the four percentiles.

The table shows that IQB has strong concurrent validity at P25 and P50, and weak validity at the framework default. At P50 the correlation with GDP per capita is about 0.8 (0.77), and about 0.7 with fixed-broadband penetration and Internet use; at P25 it is only slightly lower. At P95 these fall to about 0.5. That is not because P95 is "wrong", but because it pushes many countries to the ceiling (35% of these countries score exactly 1.0 at P95), leaving too little variation to correlate. ISP concentration is weakly *negatively* associated with IQB at P50 and above (ρ ≈ −0.2 to −0.3), but the section on information beyond income shows this does not survive controls.

## Which percentile validates best?

In [ ]:
prof = pd.DataFrame(
    {label: [spearmanr(S[q], d[ind], nan_policy="omit").statistic for q in T.Q_GRID] for ind, label in INDICATORS.items()},
    index=T.Q_GRID,
)
best = prof.idxmax()
fig, ax = plt.subplots(figsize=(8.5, 4))
show = ["GDP per capita (log, PPP)", "Fixed broadband per 100 (ITU)", "Internet users, % (ITU)"]
for c, label in zip(T.SERIES, show):
    ax.plot(prof.index, prof[label], color=c, label=label)
ax.plot(prof.index, -prof["ISP HHI (APNIC)"], color=T.SERIES[3], label="ISP HHI (APNIC), sign flipped")
ax.axvline(95, color=T.AXIS, lw=0.8)
ax.text(94, 0.04, "framework default P95", ha="right", color=T.MUTED, fontsize=9)
ax.axvline(50, color=T.AXIS, lw=0.8)
ax.text(51, 0.04, "P50", color=T.MUTED, fontsize=9)
ax.set_xlim(0, 100)
ax.set_ylim(0, 1)
ax.set_xlabel("percentile used for scoring")
ax.set_ylabel("Spearman ρ with indicator")
ax.set_title("Concurrent validity across the percentile range (2025)")
ax.legend(loc="upper left", fontsize=8)
T.savefig(fig, "4b_validity_by_percentile")
print("percentile with the highest correlation:")
print(best.to_string())

The figure repeats the correlation for every scoring percentile from P1 to P99. The x-axis is the percentile used to compute IQB, the y-axis the Spearman ρ between that IQB and the indicator, and each line is one indicator; ISP concentration is drawn with its sign flipped, so that higher means a stronger negative association. The vertical lines mark P50 and the framework default, P95. The printout gives, for each indicator, the percentile at which the correlation peaks.

The three development indicators move together. Their correlation climbs steeply over the first few percentiles, peaks at about 0.75–0.8 around P40 (the printed peaks are at P40–P42), stays around 0.7 until about P85 and then falls to about 0.4–0.45 at P99. The flipped ISP HHI line stays low throughout and rises slowly with the percentile, to about 0.3 above P80. Its printed peak at P1 only means that the correlation there is slightly positive (0.12) rather than negative.

The figure shows that validity peaks around P40–P50 for GDP, ITU broadband, Internet use and secure servers (mobile subscriptions, the weakest indicator, peaks higher, at P75), and declines steadily toward P95. If IQB is meant to reflect connectivity as experienced by the population, the evidence favors a percentile near the median.

## What do the relationships look like?

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6), sharey=True)
pairs = [("log10_gdp_pc", "GDP per capita (log10, PPP $)"), ("fixed_bb_per100", "fixed broadband per 100"),
         ("internet_users_pct", "Internet users (%)"), ("isp_hhi", "ISP HHI")]
for ax, (ind, label) in zip(axes, pairs):
    for inc in T.INCOME_ORDER:
        x = d[d.income == inc]
        ax.scatter(x[ind], S.loc[x.index, 50], s=16, color=T.INCOME_COLORS[inc], label=T.INCOME_SHORT[inc],
                   edgecolor=T.SURFACE, linewidth=0.8)
    ax.set_xlabel(label)
    ax.set_title(f"ρ = {spearmanr(S[50], d[ind], nan_policy='omit').statistic:.2f}")
axes[0].set_ylabel("IQB at P50 (2025)")
fig.legend(*axes[0].get_legend_handles_labels(), title="income", loc="upper right", ncol=4, bbox_to_anchor=(0.99, 1.02))
fig.suptitle("IQB at P50 against external indicators (dots: countries with an income group; ρ: all countries with the indicator)", x=0.01, ha="left", fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.92))
T.savefig(fig, "4b_scatter_indicators")

Each panel plots IQB at P50 (y-axis, the same in every panel) against one indicator (x-axis): GDP per capita on a log scale, fixed-broadband subscriptions per 100 people, the share of the population using the Internet, and ISP market concentration (HHI, from near 0 for a market with many small operators to 10,000 for a monopoly). Each dot is a country with an income group, colored by income; the ρ above each panel is computed over all countries with the indicator.

In the first three panels the dots rise from lower left to upper right, broadly in income order, but with a wide spread at every level: among countries with GDP per capita around 10^4.0–10^4.5 (about $10,000–$30,000), scores range from 0 to about 0.95. Scores of exactly 0 occur across a wide range of each indicator, including in some upper-middle- and high-income countries. The HHI panel shows no clear pattern, and most countries lie between about 1,000 and 5,000.

## Does IQB carry information beyond income?

We compute partial Spearman correlations, first with GDP per capita held fixed, then with GDP and the number of tests (log) held fixed. We control for test volume because it differs systematically between markets and is itself associated with IQB (ρ = 0.38 at P50, 0.54 at P95).

In [ ]:
rows = []
for q in (25, 50, 75, 95):
    for ind, label in INDICATORS.items():
        if ind == "log10_gdp_pc":
            continue
        r1, p1, n1 = T.partial_spearman(S[q], d[ind], {"gdp": d.log10_gdp_pc})
        r2, p2, n2 = T.partial_spearman(S[q], d[ind], {"gdp": d.log10_gdp_pc, "tests": d.log10_tests})
        rows.append((f"P{q}", label, spearmanr(S[q], d[ind], nan_policy="omit").statistic, r1, p1, r2, p2, n2))
partial = pd.DataFrame(rows, columns=["level", "indicator", "rho", "partial | GDP", "p", "partial | GDP, tests", "p ", "n"])
partial.set_index(["level", "indicator"]).round(3)

Each row is an indicator at one percentile; GDP per capita is left out because it is the control. *rho* is the plain Spearman correlation with IQB, as in the first table. *partial | GDP* is the correlation that remains once GDP per capita is held fixed, that is, the correlation between the parts of IQB and of the indicator that GDP does not explain, with its p-value in *p*. *partial | GDP, tests* also holds the number of tests (log) fixed, with its p-value in the second *p* column, and *n* is the number of countries. A p-value below 0.05 means the remaining association is unlikely to be chance.

Holding GDP fixed shrinks every correlation: the largest partial correlation is 0.44, against plain correlations of up to 0.78. Secure servers keep the largest partial correlation at every percentile (0.28–0.35 with both controls, all with p < 0.001), and ISP HHI's partial correlation drops to about 0 once the number of tests is also held fixed.

In [ ]:
within = []
for inc in T.INCOME_ORDER:
    x = d[d.income == inc]
    within.append({"income": T.INCOME_SHORT[inc], "countries": len(x),
                   **{label: spearmanr(S.loc[x.index, 50], x[ind], nan_policy="omit").statistic
                      for ind, label in INDICATORS.items()}})
print("Spearman ρ with IQB at P50, within income groups:")
pd.DataFrame(within).set_index("income").round(2)

Each value is the Spearman correlation between IQB at P50 and an indicator (columns) among the countries of one income group (rows), and *countries* gives the size of the group. Comparing countries of similar income removes most of the variation behind the correlations in the first table, so these are much weaker, between −0.38 and 0.53. In low-income countries most are negative, though with 22 countries none is precise. Secure servers is the most consistent indicator: positive in every group, and highest for upper-middle- and high-income countries (0.46 and 0.52).

Beyond income, IQB carries little independent information about the external indicators.

For Internet users and mobile subscriptions, the association at P50 and P95 vanishes once GDP per capita is held fixed: both largely reflect income. The exceptions are small. At P75 Internet use keeps a partial ρ of about 0.2 (0.18 with test volume also fixed), and at P25 mobile subscriptions turn slightly *negative* (about −0.2).

Fixed-broadband penetration has a small positive partial association with GDP held fixed (about 0.2 at every percentile). Once test volume is also held fixed it is no longer significant at P50–P95, though it remains at P25 (0.18).

ISP concentration has a negative partial association with GDP held fixed at P50–P95 (−0.17 to −0.30), which **disappears once test volume is held fixed** (about 0 at every percentile). Concentrated markets produce fewer M-Lab tests (ρ = −0.57 between HHI and test volume), and fewer tests go with lower scores, so there is no evidence of an independent link between market concentration and IQB.

Secure-server density is the one indicator that keeps a clear association at every percentile (partial ρ 0.28–0.35 with GDP and test volume held fixed). It is a proxy for local hosting infrastructure: content and measurement servers close to users mean lower latency and loss, which IQB rewards.

Within income groups, correlations are weak and mixed, as expected when the range of development is narrowed, and the groups are small.

## Does IQB at p50 track indicators better than a plain median download speed? 

In [ ]:
rows = []
for ind, label in INDICATORS.items():
    r_iqb = spearmanr(S[50], d[ind], nan_policy="omit").statistic
    r_dl = spearmanr(d.median_download, d[ind], nan_policy="omit").statistic
    p_iqb, pp_iqb, _ = T.partial_spearman(
        S[50], d[ind], {"dl": np.log10(d.median_download)}
    )
    p_dl, pp_dl, _ = T.partial_spearman(d.median_download, d[ind], {"iqb": S[50]})
    rows.append((label, r_iqb, r_dl, p_iqb, pp_iqb, p_dl, pp_dl))
pd.DataFrame(
    rows,
    columns=[
        "indicator",
        "ρ IQB P50",
        "ρ median download",
        "IQB | download",
        "p",
        "download | IQB",
        "p ",
    ],
).set_index("indicator").round(3)


Each row is an indicator. The first two columns are the plain correlations of IQB at P50 and of the median download speed with that indicator. *IQB | download* is the correlation of IQB with the indicator once the median download speed (log) is held fixed, and *download | IQB* the correlation of the median download with the indicator once IQB is held fixed; each is followed by its p-value. A partial correlation well away from 0 means the measure carries information about the indicator that the other one lacks.

The plain correlations are close for every indicator except ISP HHI (within about 0.06), with median download slightly ahead for GDP and fixed broadband and IQB slightly ahead for secure servers. Both partial columns are mostly between 0.1 and 0.45 in size, so each measure keeps some information the other lacks.

In [ ]:
print(
    f"Spearman(IQB P50, median download) = {spearmanr(S[50], d.median_download).statistic:.2f}; "
    f"Spearman(IQB P50, median latency) = {spearmanr(S[50], -d.median_latency).statistic:.2f}"
)

The printout gives the rank correlation of IQB at P50 with the median download speed (0.87) and with the median latency (0.78; latency is negated so that higher means better). IQB and median download therefore rank countries similarly, and they track development equally well (ρ with GDP 0.77 vs 0.80). Each keeps some association with the indicators once the other is held fixed (partial ρ 0.2–0.4). IQB's distinct part comes from latency and loss: it is more closely tied to secure-server density than speed is. So IQB is not merely a relabeled speed ranking, but it is not a *better* development indicator either. Its main added value is **interpretability** (which activities are supported) and its coverage of latency and loss.

## Does data sufficiency affect validity?

In [ ]:
allc = C_all.join(ctx[list(INDICATORS)])
S_all = T.iqb_scores(allc, cfg, [50])
allc["n_binding"] = T.binding_tests(allc)  # bands by the smaller of download and upload tests
rows = []
for lo, hi, label in [(0, T.MIN_TESTS, f"below the minimum (< {T.MIN_TESTS:,})"),
                      (T.MIN_TESTS, 12_000, f"{T.MIN_TESTS:,}–12,000"), (12_000, 100_000, "12,000–100,000"),
                      (100_000, np.inf, "≥ 100,000")]:
    sel = (allc.n_binding >= lo) & (allc.n_binding < hi)
    r = spearmanr(S_all.loc[sel, 50], allc.loc[sel, "log10_gdp_pc"], nan_policy="omit")
    rows.append((label, int(sel.sum()), int(allc.loc[sel, "log10_gdp_pc"].notna().sum()), r.statistic))
pd.DataFrame(rows, columns=["2025 test volume", "countries", "with GDP", "ρ(IQB P50, GDP)"]).set_index("2025 test volume").round(2)

The table groups all countries with tests in 2025, including those below the minimum, by their 2025 test volume (the smaller of download and upload tests). For each group it gives the number of countries, how many of them have a GDP value, and the Spearman correlation between IQB at P50 and GDP per capita among those. Only 14 of the 34 countries below the minimum have a GDP value, so that correlation rests on very few countries. The correlation rises with test volume, from 0.21 below the minimum to 0.53 between 1,153 and 12,000 tests and about 0.8 above 12,000.

The table shows that countries with little data have scores that track development much less well (ρ with GDP 0.21 below the minimum, 0.53 between 1,153 and 12,000 tests, about 0.8 above that), consistent with the sampling noise documented in 1a. The low-volume groups are small, so these correlations are imprecise. Validity analyses, and any policy use, should apply the minimum number of tests from 1a.

## Save

In [ ]:
corr.to_csv(T.TAB / "indicator_correlations.csv", index=False)
partial.to_csv(T.TAB / "indicator_partial_correlations.csv", index=False)
prof.to_csv(T.TAB / "validity_by_percentile.csv")
print("saved indicator_correlations.csv, indicator_partial_correlations.csv and validity_by_percentile.csv to", T.TAB)

## Summary

| Question | Result |
|---|---|
| Does IQB correlate with GDP and ITU indicators? | **Yes, strongly at P25 and P50** (ρ ≈ 0.8 with GDP at P50, ≈ 0.7 with broadband and Internet use); **weakly at the default P95** (≈ 0.5), because of the ceiling. |
| Which percentile is most valid? | **Around P40–P50** for the main development indicators. |
| Does IQB carry information beyond income? | **Little.** Only secure-server density (local hosting infrastructure) keeps a clear association once GDP and test volume are held fixed. |
| Is IQB more valid than median download speed? | **Equally valid, partly different.** It adds latency and loss, and its main value is interpretability. |
| Market concentration (HHI) | Weak negative raw association with IQB that **disappears once test volume is held fixed**. |

For Pulse Country Reports, IQB at **P50** is therefore defensible as a validated summary of typical connectivity. P95 should be presented as "best-case capacity", not as the headline. Scores should carry a data-sufficiency flag (1a) and a note that the score agrees with, but does not go beyond, simpler speed measures as an indicator of development.